In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("sachin_perfomance.csv")

In [3]:
print("Initial Shape:", df.shape)

Initial Shape: (782, 16)


In [4]:
df.head()

,Unnamed: 0.1,Unnamed: 0,runs,mins,BF,4s,6s,SR,Pos,Dismissal,Inns,opponent,ground,date,match,total
0,0,0,15,28,24,2,0,62.50,6,bowled,2,Pakistan,Karachi,15Nov1989,Test,15
1,1,2,59,254,172,4,0,34.30,6,lbw,1,Pakistan,Faisalabad,23Nov1989,Test,74
2,2,3,8,24,16,1,0,50.00,6,runout,3,Pakistan,Faisalabad,23Nov1989,Test,82
3,3,4,41,124,90,5,0,45.55,7,bowled,1,Pakistan,Lahore,1Dec1989,Test,123
4,4,5,35,74,51,5,0,68.62,6,lbw,1,Pakistan,Sialkot,9Dec1989,Test,158


In [5]:
# 3. Remove Unnecessary Columns
df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

if df.columns[0].startswith('Unnamed'):
    df.drop(columns=[df.columns[0]], inplace=True)

print(df.columns)


Index(['runs', 'mins', 'BF', '4s', '6s', 'SR', 'Pos', 'Dismissal', 'Inns',
       'opponent', 'ground', 'date', 'match', 'total'],
      dtype='object')


In [6]:
df.head()

,runs,mins,BF,4s,6s,SR,Pos,Dismissal,Inns,opponent,ground,date,match,total
0,15,28,24,2,0,62.50,6,bowled,2,Pakistan,Karachi,15Nov1989,Test,15
1,59,254,172,4,0,34.30,6,lbw,1,Pakistan,Faisalabad,23Nov1989,Test,74
2,8,24,16,1,0,50.00,6,runout,3,Pakistan,Faisalabad,23Nov1989,Test,82
3,41,124,90,5,0,45.55,7,bowled,1,Pakistan,Lahore,1Dec1989,Test,123
4,35,74,51,5,0,68.62,6,lbw,1,Pakistan,Sialkot,9Dec1989,Test,158


In [7]:
# 4. Standardize Column Names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.replace(r'\s+', '_', regex=True)
    .str.replace(r'_+', '_', regex=True)
)

print("Cleaned Columns:", df.columns.tolist())



Cleaned Columns: ['runs', 'mins', 'bf', '4s', '6s', 'sr', 'pos', 'dismissal', 'inns', 'opponent', 'ground', 'date', 'match', 'total']


In [8]:
# 5. Clean Text Values
for col in df.select_dtypes(include='object').columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace('_', ' ')
        .str.lower()
    )


In [9]:
# 6. Handle Missing Values
df['mins'] = df['mins'].replace('-', np.nan)
df['sr'] = df['sr'].replace('-', np.nan)

In [10]:
# 7. Clean Runs Column and Not Out Flag
df['not_out_flag'] = df['runs'].astype(str).str.contains('\*').astype(int)

df['runs'] = df['runs'].astype(str).str.replace('*', '', regex=False)
df['runs'] = pd.to_numeric(df['runs'], errors='coerce')

In [11]:
# 8. Convert Data Types
numeric_cols = ['runs', 'mins', 'bf', '4s', '6s', 'sr', 'pos', 'inns', 'total']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [12]:
df.head()

,runs,mins,bf,4s,6s,sr,pos,dismissal,inns,opponent,ground,date,match,total,not_out_flag
0,15,28.0,24.0,2.0,0,62.50,6,bowled,2,pakistan,karachi,15nov1989,test,15,0
1,59,254.0,172.0,4.0,0,34.30,6,lbw,1,pakistan,faisalabad,23nov1989,test,74,0
2,8,24.0,16.0,1.0,0,50.00,6,runout,3,pakistan,faisalabad,23nov1989,test,82,0
3,41,124.0,90.0,5.0,0,45.55,7,bowled,1,pakistan,lahore,1dec1989,test,123,0
4,35,74.0,51.0,5.0,0,68.62,6,lbw,1,pakistan,sialkot,9dec1989,test,158,0


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 782 entries, 0 to 781
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   runs          782 non-null    int64  
 1   mins          728 non-null    float64
 2   bf            781 non-null    float64
 3   4s            781 non-null    float64
 4   6s            782 non-null    int64  
 5   sr            780 non-null    float64
 6   pos           782 non-null    int64  
 7   dismissal     782 non-null    object 
 8   inns          782 non-null    int64  
 9   opponent      782 non-null    object 
 10  ground        782 non-null    object 
 11  date          782 non-null    object 
 12  match         782 non-null    object 
 13  total         782 non-null    int64  
 14  not_out_flag  782 non-null    int64  
dtypes: float64(4), int64(6), object(5)
memory usage: 91.8+ KB


In [14]:
# 9. Fix Date Format
def parse_date(x):
    try:
        return pd.to_datetime(x, format='%d%b%Y')
    except:
        return pd.NaT

df['date'] = df['date'].apply(parse_date)
df['date'] = pd.to_datetime(df['date'])  # ← ADD THIS LINE

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 782 entries, 0 to 781
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   runs          782 non-null    int64         
 1   mins          728 non-null    float64       
 2   bf            781 non-null    float64       
 3   4s            781 non-null    float64       
 4   6s            782 non-null    int64         
 5   sr            780 non-null    float64       
 6   pos           782 non-null    int64         
 7   dismissal     782 non-null    object        
 8   inns          782 non-null    int64         
 9   opponent      782 non-null    object        
 10  ground        782 non-null    object        
 11  date          782 non-null    datetime64[ns]
 12  match         782 non-null    object        
 13  total         782 non-null    int64         
 14  not_out_flag  782 non-null    int64         
dtypes: datetime64[ns](1), float64(4), int64(

In [16]:
# 10. Standardize Team Names
team_map = {
    'wi': 'west indies',
    'westindies': 'west indies',
    'aus': 'australia',
    'nz': 'new zealand',
    'sl': 'sri lanka'
}

df['opponent'] = df['opponent'].replace(team_map)

In [17]:
opponent_fix = {
    'newzealand': 'new zealand',
    'southafrica': 'south africa',
    'srilanka': 'sri lanka',
    'u.a.e.': 'uae'
}
df['opponent'] = df['opponent'].replace(opponent_fix)

In [18]:
# 11. Standardize Other Categorical Columns
df['dismissal'] = df['dismissal'].str.strip().str.lower()
df['dismissal'] = df['dismissal'].replace({
    'runout': 'run out',
    'notout': 'not out',
    'hitwicket': 'hit wicket',
    'retirednotout': 'retired not out'
})
df['ground'] = df['ground'].str.strip()

In [19]:
# 12. Remove Invalid Values
df = df[df['runs'] >= 0]
df = df[df['sr'].isna() | (df['sr'].between(0, 1000))]
df = df[df['bf'].isna() | (df['bf'] > 0)]

In [20]:
# 13. Create Year and Month
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

In [21]:
# 14. Home / Away Classification
home_grounds = ['delhi', 'kolkata', 'chennai', 'mumbai', 
                'hyderabad', 'bengaluru', 'nagpur', 'pune',
                'chandigarh', 'margao', 'cuttack', 'mohali',
                'ahmedabad', 'jaipur', 'kanpur', 'indore']
df['home'] = df['ground'].apply(
    lambda x: 'Home' if any(city in str(x).lower() for city in home_grounds) else 'Away'
)

In [22]:
# 15. Match Type Standardization
df['match_type'] = df['match'].map({
    'odi': 'odi',
    'test': 'test',
    'ti': 't20i'
})
df.drop(columns=['match'], inplace=True)

In [23]:
# 16. Career Phase Classification
def career_phase(year):
    if pd.isna(year):
        return np.nan
    elif year < 1998:
        return 'Early Career'
    elif year < 2007:
        return 'Prime Career'
    else:
        return 'Late Career'

df['career_phase'] = df['year'].apply(career_phase)

In [24]:
# 17. Create 50s and 100s Flags
df['is_50'] = df['runs'].apply(lambda x: 1 if pd.notna(x) and 50 <= x < 100 else 0)
df['is_100'] = df['runs'].apply(lambda x: 1 if pd.notna(x) and x >= 100 else 0)

In [25]:
# 18. Validate Strike Rate
df['calculated_sr'] = (df['runs'] / df['bf']) * 100

In [26]:
# 19. Remove Duplicates
df.drop_duplicates(inplace=True)

In [27]:
# 20. Final Validation Checks
print("Final Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())
print("\nData Types:\n", df.dtypes)

Final Shape: (781, 22)

Missing Values:
 runs              0
mins             54
bf                1
4s                1
6s                0
sr                1
pos               0
dismissal         0
inns              0
opponent          0
ground            0
date              0
total             0
not_out_flag      0
year              0
month             0
home              0
match_type        0
career_phase      0
is_50             0
is_100            0
calculated_sr     1
dtype: int64

Data Types:
 runs                      int64
mins                    float64
bf                      float64
4s                      float64
6s                        int64
sr                      float64
pos                       int64
dismissal                object
inns                      int64
opponent                 object
ground                   object
date             datetime64[ns]
total                     int64
not_out_flag              int64
year                      int32
month       

In [28]:
# 21 Check rows with remaining missing values
df[df[['bf', '4s', 'sr', 'calculated_sr']].isnull().any(axis=1)]

,runs,mins,bf,4s,6s,sr,pos,dismissal,inns,opponent,...,total,not_out_flag,year,month,home,match_type,career_phase,is_50,is_100,calculated_sr
22,11,92.0,NaN,NaN,0,NaN,6,lbw,1,sri lanka,...,704,0,1990,11,Home,test,Early Career,0,0,NaN


In [29]:
# 22 Fill boundary statistics where missing
df['4s'] = df['4s'].fillna(0)
df['6s'] = df['6s'].fillna(0)
df['4s'] = df['4s'].astype(int)  # ← ADD THIS

In [30]:
# 23 Data Quality checking
df['data_quality'] = df['bf'].apply(lambda x: 'complete' if pd.notna(x) else 'partial')

In [31]:
print("Final Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())

Final Shape: (781, 23)

Missing Values:
 runs              0
mins             54
bf                1
4s                0
6s                0
sr                1
pos               0
dismissal         0
inns              0
opponent          0
ground            0
date              0
total             0
not_out_flag      0
year              0
month             0
home              0
match_type        0
career_phase      0
is_50             0
is_100            0
calculated_sr     1
data_quality      0
dtype: int64


In [32]:
df = df.sort_values(by=['date', 'inns'])

In [33]:
# Final validation
print("Final Shape:", df.shape)

# Fix date format
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['date', 'inns'])

# Final improvements
df.rename(columns={'4s': 'fours', '6s': 'sixes'}, inplace=True)
df['calculated_sr'] = df['calculated_sr'].round(2)

# Export
df.to_csv('sachin_tendulkar_cleaned.csv', index=False)
print("✅ Final cleaned dataset ready")

Final Shape: (781, 23)
✅ Final cleaned dataset ready


In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 781 entries, 0 to 781
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   runs           781 non-null    int64         
 1   mins           727 non-null    float64       
 2   bf             780 non-null    float64       
 3   fours          781 non-null    int64         
 4   sixes          781 non-null    int64         
 5   sr             780 non-null    float64       
 6   pos            781 non-null    int64         
 7   dismissal      781 non-null    object        
 8   inns           781 non-null    int64         
 9   opponent       781 non-null    object        
 10  ground         781 non-null    object        
 11  date           781 non-null    datetime64[ns]
 12  total          781 non-null    int64         
 13  not_out_flag   781 non-null    int64         
 14  year           781 non-null    int32         
 15  month          781 non-null 

In [35]:
from sqlalchemy import create_engine
import urllib

# Your SQL Server details
server = r"JYOTHSNA"   # Use raw string for backslash
database = "sachin_career_analysis_project"            # Your database name
driver = "ODBC Driver 17 for SQL Server"   # Must be installed on your machine

# Build connection string for Windows Authentication
connection_string = urllib.parse.quote_plus(
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
)

# Create engine
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={connection_string}")

print("✔ Successfully connected to SQL Server (Windows Authentication)!")

✔ Successfully connected to SQL Server (Windows Authentication)!


In [36]:
table_name = "sachin_career"   # SQL table name you want to create

df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"✔ Data successfully loaded into table '{table_name}' in database 'sachin_career_analysis_project.")

C:\Users\jyoth\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


✔ Data successfully loaded into table 'sachin_career' in database 'sachin_career_analysis_project.
